# Notebook 02 — Supervised 10% Deformable DETR Baseline (150 epochs, standardized) — Standardized

Notebook này là supervised baseline sạch cho chuỗi thí nghiệm:

- Chỉ dùng **10% labeled training images**.
- Không dùng unlabeled data để train.
- Không dùng Teacher–Student, pseudo-label, SHM, CQC hoặc CPM.
- Cùng seed/split/model/data pipeline với NB03A và NB03B.
- Mốc 150 epochs dùng để kiểm tra rõ hơn việc mô hình đã plateau hay chưa.

In [ ]:
# Chạy cell này trên Kaggle/Colab nếu môi trường thiếu package.
import sys, subprocess, importlib.util

REQUIRED_PACKAGES = [
    ("transformers", "transformers==4.40.2"),
    ("timm", "timm"),
    ("pycocotools", "pycocotools"),
    ("albumentations", "albumentations>=1.3.0"),
]

def ensure_package(import_name, pip_name):
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

for import_name, pip_name in REQUIRED_PACKAGES:
    ensure_package(import_name, pip_name)

print("Dependencies are ready")

In [ ]:
import os, io, json, time, math, copy, random, contextlib, warnings, tempfile
from pathlib import Path
from collections import Counter, defaultdict

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ["HF_TOKEN"] = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"  # --- IGNORE ---
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from transformers import AutoImageProcessor, DeformableDetrConfig, DeformableDetrForObjectDetection
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

try:
    from torchvision.ops import nms
except Exception:
    nms = None

print(f"Torch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def resolve_data_root(candidates):
    for cand in candidates:
        root = Path(cand)
        if (root / "train/images").exists() and (root / "train/labels").exists():
            return root
    raise FileNotFoundError(
        "Không tìm thấy data_root hợp lệ. Hãy sửa CFG['data_root_candidates'] cho đúng Kaggle dataset path.\n"
        + "Candidates:\n" + "\n".join(map(str, candidates))
    )


def scan_split(data_root: Path, split: str):
    img_dir = data_root / split / "images"
    lbl_dir = data_root / split / "labels"
    if not img_dir.exists():
        raise FileNotFoundError(f"Missing image folder: {img_dir}")
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    img_paths = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in exts])
    if len(img_paths) == 0:
        raise RuntimeError(f"No images found in: {img_dir}")
    return [(p, lbl_dir / f"{p.stem}.txt") for p in img_paths]


def split_labeled_unlabeled(pairs, ratio: float, seed: int):
    rng = random.Random(seed)
    pairs = list(pairs)
    rng.shuffle(pairs)
    n_labeled = max(1, int(round(len(pairs) * ratio)))
    return pairs[:n_labeled], pairs[n_labeled:]


def read_raw_yolo_classes(label_path: Path):
    classes = []
    if label_path.exists():
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    classes.append(int(float(parts[0])))
    return classes


def infer_label_offset(pairs, num_classes: int):
    raw = []
    for _, lbl in pairs:
        raw.extend(read_raw_yolo_classes(lbl))
    if not raw:
        return 0, {"min": None, "max": None, "counter": {}}
    mn, mx = min(raw), max(raw)
    # Hỗ trợ cả YOLO class 0..C-1 và 1..C.
    offset = 1 if (mn == 1 and mx == num_classes) else 0
    return offset, {"min": int(mn), "max": int(mx), "counter": dict(Counter(raw))}


def load_yolo_label(label_path: Path, img_w: int, img_h: int, label_offset: int, num_classes: int):
    boxes, labels = [], []
    if label_path.exists():
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls = int(float(parts[0])) - label_offset
                if cls < 0 or cls >= num_classes:
                    continue
                cx, cy, bw, bh = map(float, parts[1:5])
                x1 = (cx - bw / 2.0) * img_w
                y1 = (cy - bh / 2.0) * img_h
                x2 = (cx + bw / 2.0) * img_w
                y2 = (cy + bh / 2.0) * img_h
                x1, y1 = max(0.0, x1), max(0.0, y1)
                x2, y2 = min(float(img_w), x2), min(float(img_h), y2)
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(cls)
    return np.asarray(boxes, dtype=np.float32), np.asarray(labels, dtype=np.int64)


def make_pad_if_needed(img_size: int):
    # Albumentations 1.x dùng value, 2.x ưu tiên fill. Try/except để notebook chạy ổn trên nhiều Kaggle image.
    try:
        return A.PadIfNeeded(
            min_height=img_size,
            min_width=img_size,
            border_mode=cv2.BORDER_CONSTANT,
            value=0,
        )
    except TypeError:
        return A.PadIfNeeded(
            min_height=img_size,
            min_width=img_size,
            border_mode=cv2.BORDER_CONSTANT,
            fill=0,
        )


def make_coarse_dropout():
    try:
        return A.CoarseDropout(max_holes=8, max_height=64, max_width=64, p=0.25)
    except TypeError:
        return A.CoarseDropout(
            num_holes_range=(1, 8),
            hole_height_range=(8, 64),
            hole_width_range=(8, 64),
            p=0.25,
        )


def get_transform(img_size: int, train: bool):
    # Không dùng rotation thủ công vì dễ làm sai bbox. Tất cả transform hình học đi qua Albumentations bbox-aware.
    transforms = [
        A.LongestMaxSize(max_size=img_size),
        make_pad_if_needed(img_size),
    ]
    if train:
        transforms.extend([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.2),
            A.RandomBrightnessContrast(p=0.35),
            A.HueSaturationValue(p=0.20),
        ])
    return A.Compose(
        transforms,
        bbox_params=A.BboxParams(
            format="pascal_voc",
            label_fields=["class_labels"],
            min_visibility=0.10,
            check_each_transform=True,
        ),
    )


def get_strong_photometric_aug():
    # Chuẩn hóa across NB03A/NB04/NB05/NB06: PCB defects thường nhỏ; aggressive blur/cutout
    # có thể làm mất defect nhưng pseudo-box vẫn còn. Giữ strong view chủ yếu là photometric nhẹ.
    return A.Compose([
        A.RandomBrightnessContrast(brightness_limit=0.20, contrast_limit=0.20, p=0.55),
        A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=20, val_shift_limit=15, p=0.25),
        A.GaussNoise(p=0.10),
        A.MotionBlur(blur_limit=3, p=0.05),
    ])

def xyxy_to_cxcywh_norm(boxes, width: int, height: int):
    if len(boxes) == 0:
        return np.zeros((0, 4), dtype=np.float32)
    boxes = np.asarray(boxes, dtype=np.float32)
    cx = (boxes[:, 0] + boxes[:, 2]) / 2.0 / width
    cy = (boxes[:, 1] + boxes[:, 3]) / 2.0 / height
    bw = (boxes[:, 2] - boxes[:, 0]) / width
    bh = (boxes[:, 3] - boxes[:, 1]) / height
    return np.clip(np.stack([cx, cy, bw, bh], axis=1), 0.0, 1.0).astype(np.float32)


def cxcywh_norm_to_xyxy_abs(boxes, img_size: int):
    boxes = np.asarray(boxes, dtype=np.float32)
    if len(boxes) == 0:
        return np.zeros((0, 4), dtype=np.float32)
    cx, cy, bw, bh = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    x1 = (cx - bw / 2.0) * img_size
    y1 = (cy - bh / 2.0) * img_size
    x2 = (cx + bw / 2.0) * img_size
    y2 = (cy + bh / 2.0) * img_size
    out = np.stack([x1, y1, x2, y2], axis=1)
    return np.clip(out, 0.0, float(img_size)).astype(np.float32)


class YoloDetectionDataset(Dataset):
    def __init__(self, pairs, img_size, transform, label_offset, num_classes, mode="train"):
        self.pairs = list(pairs)
        self.img_size = int(img_size)
        self.transform = transform
        self.label_offset = int(label_offset)
        self.num_classes = int(num_classes)
        self.mode = mode

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, lbl_path = self.pairs[idx]
        image_bgr = cv2.imread(str(img_path))
        if image_bgr is None:
            raise FileNotFoundError(img_path)
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        h0, w0 = image.shape[:2]
        boxes, labels = load_yolo_label(lbl_path, w0, h0, self.label_offset, self.num_classes)
        aug = self.transform(image=image, bboxes=boxes.tolist(), class_labels=labels.tolist())
        image = aug["image"]
        boxes = np.asarray(aug["bboxes"], dtype=np.float32)
        labels = np.asarray(aug["class_labels"], dtype=np.int64)
        h, w = image.shape[:2]
        boxes_norm = xyxy_to_cxcywh_norm(boxes, w, h)
        return {
            "image": image,
            "boxes": boxes_norm,
            "labels": labels,
            "image_id": int(idx),
            "img_path": str(img_path),
        }


class YoloUnlabeledDataset(Dataset):
    def __init__(self, pairs, img_size, weak_transform, strong_photo_aug):
        self.pairs = list(pairs)
        self.img_size = int(img_size)
        self.weak_transform = weak_transform
        self.strong_photo_aug = strong_photo_aug

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, _ = self.pairs[idx]
        image_bgr = cv2.imread(str(img_path))
        if image_bgr is None:
            raise FileNotFoundError(img_path)
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        weak_aug = self.weak_transform(image=image, bboxes=[], class_labels=[])
        weak = weak_aug["image"]
        strong = self.strong_photo_aug(image=weak)["image"]
        return {
            "weak_image": weak,
            "strong_image": strong,
            "image_id": int(idx),
            "img_path": str(img_path),
        }


def sample_to_coco_annotations(item, img_size: int, ann_id_start=1):
    anns = []
    for j, (box, cls) in enumerate(zip(item["boxes"], item["labels"])):
        cx, cy, bw, bh = map(float, box)
        x1 = max(0.0, (cx - bw / 2.0) * img_size)
        y1 = max(0.0, (cy - bh / 2.0) * img_size)
        x2 = min(float((cx + bw / 2.0) * img_size), float(img_size))
        y2 = min(float((cy + bh / 2.0) * img_size), float(img_size))
        w, h = x2 - x1, y2 - y1
        if w <= 1 or h <= 1:
            continue
        anns.append({
            "id": int(ann_id_start + j),
            "image_id": int(item["image_id"]),
            "category_id": int(cls),
            "bbox": [float(x1), float(y1), float(w), float(h)],
            "area": float(w * h),
            "iscrowd": 0,
        })
    return anns


def seed_worker(worker_id):
    worker_seed = CFG["seed"] + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def save_split_report(path, train_pairs, labeled_pairs, unlabeled_pairs, valid_pairs, test_pairs):
    report = {
        "seed": CFG["seed"],
        "labeled_ratio": CFG["labeled_ratio"],
        "train_total": len(train_pairs),
        "labeled": len(labeled_pairs),
        "unlabeled": len(unlabeled_pairs),
        "valid": len(valid_pairs),
        "test": len(test_pairs),
        "labeled_files": [p[0].name for p in labeled_pairs],
        "unlabeled_files": [p[0].name for p in unlabeled_pairs],
    }
    with open(path, "w") as f:
        json.dump(report, f, indent=2)
    return report

In [ ]:
CFG = dict(
    seed=42,

    # Giữ candidate list giống Notebook 03/04/05/06. Sửa list này ở TẤT CẢ notebook nếu dataset path khác.
    data_root_candidates=[
        "/kaggle/input/datasets/hgthuhwn/my-pcb/YOLO_format",
        "/kaggle/input/datasets/nhnkhnh/my-pcb/YOLO_format",
        "/kaggle/input/datasets/minhquan228/my-pcb/YOLO_format",
        "/kaggle/input/datasets/minhhieu/datapcb-v2/YOLO_format",
        "/kaggle/input/my-pcb/YOLO_format",
        "/kaggle/input/datapcb-v2/YOLO_format",
    ],
    out_dir="/kaggle/working/artifacts/nb02_supervised_10pct_standardized_150e",

    experiment_name="nb02_supervised_10pct_deformable_detr_standardized_150e",
    model_name="SenseTime/deformable-detr",
    num_classes=5,
    class_names=["0", "1", "2", "3", "4"],
    num_queries=300,
    num_feature_levels=4,
    img_size=640,

    labeled_ratio=0.10,
    epochs=150,
    batch_size=1,
    grad_accum_steps=16,
    num_workers=2,
    pin_memory=True,
    mixed_precision=True,

    # Baseline supervised giữ LR lớn hơn SSOD fine-tuning vì train từ pretrained detector.
    lr=2e-4,
    lr_backbone=2e-5,
    weight_decay=1e-4,
    lr_milestones=[105, 135],
    lr_gamma=0.1,
    clip_grad_norm=0.1,

    eval_every_n_epochs=5,
    eval_conf_threshold=0.05,
    vis_conf_threshold=0.25,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CFG["device"] = str(DEVICE)
OUT_DIR = Path(CFG["out_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)
set_seed(CFG["seed"])

DATA_ROOT = resolve_data_root(CFG["data_root_candidates"])
CFG["data_root"] = str(DATA_ROOT)

with open(OUT_DIR / "config.json", "w") as f:
    json.dump(CFG, f, indent=2)

print(json.dumps({k: CFG[k] for k in [
    "experiment_name", "data_root", "out_dir", "labeled_ratio", "epochs", "batch_size", "grad_accum_steps",
    "lr", "lr_backbone", "lr_milestones", "eval_every_n_epochs", "device"
]}, indent=2))


In [ ]:
processor = AutoImageProcessor.from_pretrained(
    CFG["model_name"],
    size={"height": CFG["img_size"], "width": CFG["img_size"]},
    do_resize=False,
    use_fast=False,
)


def collate_labeled(batch):
    images, targets = [], []
    ann_id = 1
    for item in batch:
        images.append(Image.fromarray(item["image"]))
        anns = sample_to_coco_annotations(item, CFG["img_size"], ann_id_start=ann_id)
        ann_id += max(1, len(anns))
        targets.append({"image_id": int(item["image_id"]), "annotations": anns})
    enc = processor(images=images, annotations=targets, return_tensors="pt")
    enc["image_ids"] = torch.tensor([int(x["image_id"]) for x in batch], dtype=torch.long)
    return enc


def move_labels_to_device(labels, device):
    return [{k: v.to(device) for k, v in t.items()} for t in labels]


def build_detector(cfg):
    config = DeformableDetrConfig.from_pretrained(cfg["model_name"])
    config.num_labels = cfg["num_classes"]
    config.num_queries = cfg["num_queries"]
    config.num_feature_levels = cfg["num_feature_levels"]
    return DeformableDetrForObjectDetection.from_pretrained(
        cfg["model_name"],
        config=config,
        ignore_mismatched_sizes=True,
    )


def build_optimizer(detector, cfg):
    backbone_params, other_params = [], []
    for name, p in detector.named_parameters():
        if not p.requires_grad:
            continue
        if "backbone" in name:
            backbone_params.append(p)
        else:
            other_params.append(p)
    return torch.optim.AdamW(
        [
            {"params": other_params, "lr": cfg["lr"]},
            {"params": backbone_params, "lr": cfg["lr_backbone"]},
        ],
        weight_decay=cfg["weight_decay"],
    )


def build_scheduler(optimizer, cfg):
    return torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=list(cfg["lr_milestones"]),
        gamma=cfg["lr_gamma"],
    )


def build_coco_gt_from_dataset(dataset, name="dataset"):
    images, annotations = [], []
    ann_id = 1
    for idx in range(len(dataset)):
        item = dataset[idx]
        images.append({
            "id": int(item["image_id"]),
            "width": CFG["img_size"],
            "height": CFG["img_size"],
            "file_name": Path(item["img_path"]).name,
        })
        anns = sample_to_coco_annotations(item, CFG["img_size"], ann_id_start=ann_id)
        annotations.extend(anns)
        ann_id += max(1, len(anns))
    cats = [{"id": i, "name": str(CFG["class_names"][i])} for i in range(CFG["num_classes"])]
    coco_dict = {
        "info": {"description": name},
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": cats,
    }
    gt_path = OUT_DIR / f"gt_{name}.json"
    with open(gt_path, "w") as f:
        json.dump(coco_dict, f)
    with contextlib.redirect_stdout(io.StringIO()):
        return COCO(str(gt_path))


@torch.no_grad()
def evaluate_coco(detector, loader, dataset, device, coco_gt=None, desc="val", conf_thr=None):
    detector.eval()
    conf_thr = CFG["eval_conf_threshold"] if conf_thr is None else float(conf_thr)
    if coco_gt is None:
        with contextlib.redirect_stdout(io.StringIO()):
            coco_gt = build_coco_gt_from_dataset(dataset, name=desc)

    results = []
    for batch in loader:
        pixel_values = batch["pixel_values"].to(device)
        pixel_mask = batch.get("pixel_mask")
        pixel_mask = pixel_mask.to(device) if pixel_mask is not None else None

        with torch.amp.autocast("cuda", enabled=CFG["mixed_precision"] and device.type == "cuda"):
            outputs = detector(pixel_values=pixel_values, pixel_mask=pixel_mask)

        target_sizes = torch.tensor(
            [[CFG["img_size"], CFG["img_size"]]] * pixel_values.shape[0],
            device=device,
        )
        preds = processor.post_process_object_detection(
            outputs,
            threshold=conf_thr,
            target_sizes=target_sizes,
        )

        for img_id, pred in zip(batch["image_ids"].tolist(), preds):
            boxes = pred["boxes"].detach().cpu().numpy()
            scores = pred["scores"].detach().cpu().numpy()
            labels = pred["labels"].detach().cpu().numpy()
            for box, score, label in zip(boxes, scores, labels):
                x1, y1, x2, y2 = map(float, box)
                x1, y1 = max(0.0, min(x1, CFG["img_size"])), max(0.0, min(y1, CFG["img_size"]))
                x2, y2 = max(0.0, min(x2, CFG["img_size"])), max(0.0, min(y2, CFG["img_size"]))
                w, h = x2 - x1, y2 - y1
                if w <= 1 or h <= 1:
                    continue
                if int(label) < 0 or int(label) >= CFG["num_classes"]:
                    continue
                results.append({
                    "image_id": int(img_id),
                    "category_id": int(label),
                    "bbox": [float(x1), float(y1), float(w), float(h)],
                    "score": float(score),
                })

    empty = {"AP": 0.0, "AP50": 0.0, "AP75": 0.0, "APs": 0.0, "per_class_AP50": {}}
    if not results:
        return empty

    pred_path = OUT_DIR / f"pred_{desc}.json"
    with open(pred_path, "w") as f:
        json.dump(results, f)

    with contextlib.redirect_stdout(io.StringIO()):
        coco_dt = coco_gt.loadRes(str(pred_path))
        coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()
    stats = coco_eval.stats

    per_class_ap50 = {}
    for cat_id in coco_gt.getCatIds():
        cat_name = coco_gt.loadCats(cat_id)[0]["name"]
        with contextlib.redirect_stdout(io.StringIO()):
            coco_cls = COCOeval(coco_gt, coco_dt, "bbox")
            coco_cls.params.catIds = [cat_id]
            coco_cls.params.iouThrs = [0.5]
            coco_cls.evaluate()
            coco_cls.accumulate()
            coco_cls.summarize()
        per_class_ap50[cat_name] = float(coco_cls.stats[0])

    pc_str = "  ".join(f"{k}:{v:.3f}" for k, v in per_class_ap50.items())
    print(f"[{desc}] AP={stats[0]:.4f} AP50={stats[1]:.4f} AP75={stats[2]:.4f} APs={stats[3]:.4f} | {pc_str}")

    return {
        "AP": float(stats[0]),
        "AP50": float(stats[1]),
        "AP75": float(stats[2]),
        "APs": float(stats[3]),
        "per_class_AP50": per_class_ap50,
    }


def safe_torch_load(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)

In [ ]:
train_pairs = scan_split(DATA_ROOT, "train")
valid_pairs = scan_split(DATA_ROOT, "valid")
test_pairs  = scan_split(DATA_ROOT, "test")

LABEL_OFFSET, label_stats = infer_label_offset(train_pairs, CFG["num_classes"])
CFG["label_offset"] = LABEL_OFFSET

labeled_pairs, unlabeled_pairs = split_labeled_unlabeled(train_pairs, CFG["labeled_ratio"], CFG["seed"])
split_report = save_split_report(OUT_DIR / "split_labeled_10pct_seed42.json", train_pairs, labeled_pairs, unlabeled_pairs, valid_pairs, test_pairs)

print("=" * 72)
print("DATA SPLIT SUMMARY - BASELINE")
print("=" * 72)
print(f"Train total     : {len(train_pairs)}")
print(f"Labeled images  : {len(labeled_pairs)} ({len(labeled_pairs) / len(train_pairs) * 100:.2f}%)")
print(f"Unlabeled images: {len(unlabeled_pairs)} - not used by baseline")
print(f"Valid/Test      : {len(valid_pairs)} / {len(test_pairs)}")
print("Raw label stats :", label_stats)
print("Label offset    :", LABEL_OFFSET)
print("First 5 labeled :", [p[0].name for p in labeled_pairs[:5]])
print("=" * 72)

train_tf = get_transform(CFG["img_size"], train=True)
eval_tf  = get_transform(CFG["img_size"], train=False)

train_ds = YoloDetectionDataset(labeled_pairs, CFG["img_size"], train_tf, LABEL_OFFSET, CFG["num_classes"], mode="train")
valid_ds = YoloDetectionDataset(valid_pairs,  CFG["img_size"], eval_tf,  LABEL_OFFSET, CFG["num_classes"], mode="valid")
test_ds  = YoloDetectionDataset(test_pairs,   CFG["img_size"], eval_tf,  LABEL_OFFSET, CFG["num_classes"], mode="test")

train_loader = DataLoader(
    train_ds, batch_size=CFG["batch_size"], shuffle=True,
    num_workers=CFG["num_workers"], pin_memory=CFG["pin_memory"],
    collate_fn=collate_labeled, worker_init_fn=seed_worker, drop_last=True,
)
valid_loader = DataLoader(
    valid_ds, batch_size=max(1, CFG["batch_size"] * 2), shuffle=False,
    num_workers=CFG["num_workers"], pin_memory=CFG["pin_memory"],
    collate_fn=collate_labeled, worker_init_fn=seed_worker,
)
test_loader = DataLoader(
    test_ds, batch_size=max(1, CFG["batch_size"] * 2), shuffle=False,
    num_workers=CFG["num_workers"], pin_memory=CFG["pin_memory"],
    collate_fn=collate_labeled, worker_init_fn=seed_worker,
)

sample = train_ds[0]
print("Sample:", sample["image"].shape, sample["boxes"].shape, sample["labels"][:10])

In [ ]:
model = build_detector(CFG).to(DEVICE)
optimizer = build_optimizer(model, CFG)
scheduler = build_scheduler(optimizer, CFG)
scaler = torch.amp.GradScaler("cuda", enabled=CFG["mixed_precision"] and DEVICE.type == "cuda")

with contextlib.redirect_stdout(io.StringIO()):
    VALID_COCO_GT = build_coco_gt_from_dataset(valid_ds, name="valid_cache")

print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f}M")
print("Checkpoint criterion: best validation AP (COCO AP@[.50:.95]), not AP50")

history = []
best_ap = -1.0
best_epoch = -1
best_ckpt = OUT_DIR / "best_model.pt"
last_ckpt = OUT_DIR / "last_model.pt"

header = (
    f"{'Ep':>4} {'loss':>8} {'cls':>7} {'bbox':>7} {'giou':>7} {'card':>8} "
    f"{'lr':>10} {'grad':>7} {'time':>7} {'val_AP':>8} {'AP50':>7} {'AP75':>7} {'APs':>7}"
)
print("-" * len(header))
print(header)
print("-" * len(header))

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    t0 = time.time()
    running = defaultdict(float)
    grad_norm_sum = 0.0
    grad_steps = 0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader):
        pixel_values = batch["pixel_values"].to(DEVICE)
        pixel_mask = batch.get("pixel_mask")
        pixel_mask = pixel_mask.to(DEVICE) if pixel_mask is not None else None
        labels = move_labels_to_device(batch["labels"], DEVICE)

        with torch.amp.autocast("cuda", enabled=CFG["mixed_precision"] and DEVICE.type == "cuda"):
            outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels)
            loss = outputs.loss
            scaled_loss = loss / CFG["grad_accum_steps"]

        scaler.scale(scaled_loss).backward()

        do_step = ((step + 1) % CFG["grad_accum_steps"] == 0) or ((step + 1) == len(train_loader))
        if do_step:
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["clip_grad_norm"])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            grad_norm_sum += float(grad_norm)
            grad_steps += 1

        loss_dict = outputs.loss_dict
        running["loss"] += float(loss.detach().cpu())
        running["loss_ce"] += float(loss_dict.get("loss_ce", torch.tensor(0.0)).detach().cpu())
        running["loss_bbox"] += float(loss_dict.get("loss_bbox", torch.tensor(0.0)).detach().cpu())
        running["loss_giou"] += float(loss_dict.get("loss_giou", torch.tensor(0.0)).detach().cpu())
        running["cardinality_error"] += float(loss_dict.get("cardinality_error", torch.tensor(0.0)).detach().cpu())

    scheduler.step()
    n = max(1, len(train_loader))
    train_stats = {
        "epoch": epoch,
        "loss": running["loss"] / n,
        "loss_ce": running["loss_ce"] / n,
        "loss_bbox": running["loss_bbox"] / n,
        "loss_giou": running["loss_giou"] / n,
        "cardinality_error": running["cardinality_error"] / n,
        "lr": float(optimizer.param_groups[0]["lr"]),
        "grad_norm": grad_norm_sum / max(1, grad_steps),
        "elapsed_min": (time.time() - t0) / 60.0,
    }

    metrics = None
    saved_flag = ""
    if epoch % CFG["eval_every_n_epochs"] == 0 or epoch == CFG["epochs"]:
        metrics = evaluate_coco(model, valid_loader, valid_ds, DEVICE, coco_gt=VALID_COCO_GT, desc=f"val_ep{epoch}")
        ckpt = {
            "epoch": epoch,
            "state_dict": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "best_ap": max(best_ap, metrics["AP"]),
            "config": CFG,
            "history": history,
        }
        torch.save(ckpt, last_ckpt)
        if metrics["AP"] > best_ap:
            best_ap = metrics["AP"]
            best_epoch = epoch
            torch.save(ckpt, best_ckpt)
            saved_flag = "*"

    row = {**train_stats}
    if metrics is not None:
        row.update(metrics)
    history.append(row)

    ap = f"{metrics['AP']:.3f}" if metrics else "—"
    ap50 = f"{metrics['AP50']:.3f}" if metrics else "—"
    ap75 = f"{metrics['AP75']:.3f}" if metrics else "—"
    aps = f"{metrics['APs']:.3f}" if metrics else "—"
    print(
        f"{epoch:4d} {train_stats['loss']:8.4f} {train_stats['loss_ce']:7.4f} "
        f"{train_stats['loss_bbox']:7.4f} {train_stats['loss_giou']:7.4f} "
        f"{train_stats['cardinality_error']:8.2f} {train_stats['lr']:10.2e} "
        f"{train_stats['grad_norm']:7.2f} {train_stats['elapsed_min']:6.1f}m "
        f"{ap:>8} {ap50:>7} {ap75:>7} {aps:>7} {saved_flag}"
    )

with open(OUT_DIR / "history.json", "w") as f:
    json.dump(history, f, indent=2)

print("-" * len(header))
print(f"Training complete. Best validation AP={best_ap:.4f} at epoch {best_epoch}")

In [ ]:
ckpt = safe_torch_load(best_ckpt if best_ckpt.exists() else last_ckpt, DEVICE)
model.load_state_dict(ckpt["state_dict"])
print(f"Loaded checkpoint: epoch={ckpt['epoch']} | best_AP={ckpt.get('best_ap', best_ap):.4f}")

with contextlib.redirect_stdout(io.StringIO()):
    TEST_COCO_GT = build_coco_gt_from_dataset(test_ds, name="test_cache")

val_metrics = evaluate_coco(model, valid_loader, valid_ds, DEVICE, coco_gt=VALID_COCO_GT, desc="final_val")
test_metrics = evaluate_coco(model, test_loader, test_ds, DEVICE, coco_gt=TEST_COCO_GT, desc="final_test")

final_metrics = {
    "method": "supervised_baseline_10pct_100e",
    "best_epoch": int(ckpt["epoch"]),
    "best_val_ap": float(ckpt.get("best_ap", best_ap)),
    "val": val_metrics,
    "test": test_metrics,
    "config": CFG,
}
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(final_metrics, f, indent=2)

print(json.dumps({
    "best_epoch": final_metrics["best_epoch"],
    "val": {k: round(v, 4) for k, v in val_metrics.items() if k != "per_class_AP50"},
    "test": {k: round(v, 4) for k, v in test_metrics.items() if k != "per_class_AP50"},
}, indent=2))

In [ ]:
# Plots tối giản để kiểm tra hội tụ và metric. Có thể bỏ qua nếu chỉ cần checkpoint/metrics.json.
if len(history):
    epochs = [h["epoch"] for h in history]
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, [h["loss"] for h in history], label="total")
    plt.plot(epochs, [h["loss_ce"] for h in history], label="cls")
    plt.plot(epochs, [h["loss_bbox"] for h in history], label="bbox")
    plt.plot(epochs, [h["loss_giou"] for h in history], label="giou")
    plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.title("Baseline training losses")
    plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(OUT_DIR / "plot_losses.png", dpi=140); plt.show()

    eval_rows = [h for h in history if "AP" in h]
    if eval_rows:
        ev = [h["epoch"] for h in eval_rows]
        plt.figure(figsize=(8, 4))
        plt.plot(ev, [h["AP"] for h in eval_rows], "o-", label="AP")
        plt.plot(ev, [h["AP50"] for h in eval_rows], "s--", label="AP50")
        plt.plot(ev, [h["AP75"] for h in eval_rows], "^--", label="AP75")
        plt.xlabel("Epoch"); plt.ylabel("COCO AP")
        plt.title("Baseline validation metrics")
        plt.legend(); plt.grid(alpha=0.3)
        plt.tight_layout(); plt.savefig(OUT_DIR / "plot_val_metrics.png", dpi=140); plt.show()